In [ ]:
%run ./01-config

In [ ]:
class Upserter:
    def __init__(self, merge_query, temp_view_name):
        self.merge_query = merge_query
        self.temp_view_name = temp_view_name

    def upsert(self, df_micro_batch, batch_id):
        df_micro_batch.createOrReplaceTempView(self.temp_view_name)
        df_micro_batch._jdf.sparkSession().sql(self.merge_query)

In [ ]:
class CDCUpserter:
    def __init__(self, merge_query, temp_view_name, id_column, sort_by):
        self.merge_query = merge_query
        self.temp_view_name = temp_view_name 
        self.id_column = id_column
        self.sort_by = sort_by 
        
    def upsert(self, df_micro_batch, batch_id):
        from pyspark.sql.window import Window
        from pyspark.sql import functions as F
        
        window = Window.partitionBy(self.id_column).orderBy(F.col(self.sort_by).desc())
        
        df_micro_batch.where(F.col("update_type").isin(["new", "update"])) \
                .withColumn("rank", F.rank().over(window)).where("rank == 1").drop("rank") \
                .createOrReplaceTempView(self.temp_view_name)
        df_micro_batch._jdf.sparkSession().sql(self.merge_query)

In [ ]:
class Silver:
    def __init__(self, env):
        conf = Config()
        self.checkpoint_base = conf.base_dir_checkpoint + '/checkpoints'
        self.catalog = env
        self.db_name = conf.db_name
        spark.sql(f'use {self.catalog}.{self.db_name}')
        self.maxFilesPerTrigger = conf.maxFilesPerTrigger

    def upsert_users(self, once=True, processing_time = '5 seconds', startingVersion = 0):
        from pyspark.sql.functions import col
        from pyspark.sql.types import TimestampType
        

        query = f'''merge into {self.catalog}.{self.db_name}.users a
                    using users_delta b
                    on a.user_id = b.user_id
                    when not matched then insert *'''

        data_upserter = Upserter(query, 'users_delta')

        df_delta = (spark.readStream
                            .option('startingVersion', startingVersion)
                            .option('ignoreDeletes', True)
                            .table(f'{self.catalog}.{self.db_name}.registered_users_bz')
                            .select('user_id', 'mac_address', 'device_id', 'registration_timestamp')
                            .withColumn('registration_timestamp', col('registration_timestamp').cast(TimestampType()))
                            .withWatermark('registration_timestamp','30 seconds')
                            .dropDuplicates(['user_id', 'device_id'])
                            )
        
        stream_writer = (df_delta.writeStream
                                .outputMode('update')
                                .queryName('users_upsert_stream')
                                .option('checkpointLocation', f'{self.checkpoint_base}/users')
                                .foreachBatch(data_upserter.upsert))
        
        spark.sparkContext.setLocalProperty('spark.scheduler.pool', 'silver_p2')

        if once:
            return stream_writer.trigger(availableNow = True).start()
        else:
            return stream_writer.trigger(processingTime= processing_time).start()
        
    def upsert_gym_logs(self, once=True, processing_time = '5 seconds', startingVersion = 0):
        from pyspark.sql.functions import col
        from pyspark.sql.types import TimestampType

        query = f'''merge into {self.catalog}.{self.db_name}.gym_logs a
                    using gym_logs_delta b
                    on a.login = b.login and a.mac_address = b.mac_address and a.gym = b.gym
                    when matched and b.logout > a.login and b.logout> a.logout
                    then update set a.logout = b.logout
                    when not matched
                    then insert *'''

        data_upserter = Upserter(query, 'gym_logs_delta')

        df_delta = (spark.readStream
                            .option('ignoreDeletes', True)
                            .option('startingVersion', startingVersion)
                            .table(f'{self.catalog}.{self.db_name}.gym_logins_bz')
                            .select('mac_address', 'gym', 'login', 'logout')
                            .withColumns({'login': col('login').cast(TimestampType()), 'logout': col('logout').cast(TimestampType())})
                            .withWatermark('login', '30 seconds')
                            .dropDuplicates(['mac_address', 'gym', 'login'])
                            )
    
        writer_stream = (df_delta.writeStream
                                    .outputMode('update')
                                    .queryName('gym_logs_upsert_stream')
                                    .option('checkpointLocation', f'{self.checkpoint_base}/gym_logs')
                                    .foreachBatch(data_upserter.upsert)
                                    )

        spark.sparkContext.setLocalProperty('spark.scheduler.pool', 'silver_p3')
        
        if once:
            return writer_stream.trigger(availableNow =True).start()
        else:
            return writer_stream.trigger(processingTime = processing_time).start()


    def upsert_user_profile(self, once=False, processing_time="15 seconds", startingVersion=0):
        from pyspark.sql import functions as F

        schema = """
            user_id bigint, update_type STRING, timestamp FLOAT, 
            dob STRING, sex STRING, gender STRING, first_name STRING, last_name STRING, 
            address STRUCT<street_address: STRING, city: STRING, state: STRING, zip: INT>
            """
        
        query = f"""
            MERGE INTO {self.catalog}.{self.db_name}.user_profile a
            USING user_profile_cdc b
            ON a.user_id=b.user_id
            WHEN MATCHED AND a.updated < b.updated
              THEN UPDATE SET *
            WHEN NOT MATCHED
              THEN INSERT *
            """
        
        data_upserter = CDCUpserter(query, "user_profile_cdc", "user_id", "updated")
        
        df_cdc = (spark.readStream
                       .option("startingVersion", startingVersion)
                       .option("ignoreDeletes", True)
                       .option("withEventTimeOrder", "true")
                       .option("maxFilesPerTrigger", self.maxFilesPerTrigger)
                       .table(f"{self.catalog}.{self.db_name}.kafka_multiplex_bz")
                       .filter("topic = 'user_info'")
                       .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
                       .select("v.*")
                       .select("user_id", F.to_date('dob','MM/dd/yyyy').alias('dob'),
                               'sex', 'gender','first_name','last_name', 'address.*',
                               F.col('timestamp').cast("timestamp").alias("updated"),
                               "update_type")
                       .withWatermark("updated", "30 seconds")
                       .dropDuplicates(["user_id", "updated"])
                 )
    
        stream_writer = (df_cdc.writeStream
                               .foreachBatch(data_upserter.upsert) 
                               .outputMode("update") 
                               .option("checkpointLocation", f"{self.checkpoint_base}/user_profile") 
                               .queryName("user_profile_stream")
                        )
        
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "silver_p3")
        
        if once == True:
            return stream_writer.trigger(availableNow=True).start()
        else:
            return stream_writer.trigger(processingTime=processing_time).start()

    def upsert_heart_rate(self, once=True, processing_time = '5 seconds', startingVersion = 0):
        from pyspark.sql.functions import col, when, cast, from_json
        from pyspark.sql.types import StringType, StructType, StructField, TimestampType, LongType, DoubleType

        schema = (StructType([
                            StructField('device_id', LongType()),
                            StructField('time', TimestampType()),
                            StructField('heartrate', DoubleType())
                        ]))

        query = f'''merge into {self.catalog}.{self.db_name}.heart_rate a
                    using heart_rate_delta b
                    on a.device_id = b.device_id and a.time = b.time
                    when not matched then insert *'''

        data_upserter = Upserter(query, 'heart_rate_delta')

        df_delta = (spark.readStream
                        .option('startingVersion', startingVersion)
                        .option('ignoreDeletes', True)
                        .table(f'{self.catalog}.{self.db_name}.kafka_multiplex_bz')
                        .option('withEventTimeOrder', True)
                        .option('maxFilesPerTrigger', conf.maxFilesPerTrigger)
                        .option("badRecordsPath", f"{self.badrecords_path}/")
                        .where(col('topic') == 'bpm')
                        .withColumn('value', col('value').cast(StringType()))
                        .select(from_json(col('value'), schema).alias('val'))
                        .select('val.*')
                        .withColumn('valid', when(col('heartrate')<=0, False).otherwise(True))
                        .withWatermark('time', '30 seconds')
                        .dropDuplicates(['device_id', 'time'])
                        )
        stream_writer = (df_delta.writeStream
                                .foreachBatch(data_upserter.upsert)
                                .outputMode('update')
                                .option('checkpointLocation', f'{self.checkpoint_base}/heart_rate')
                                .queryName('heart_rate_upsert_stream'))
        
        spark.sparkContext.setLocalProperty('spark.scheduler.pool', 'silver_p2')

        if once:
            return stream_writer.trigger(availableNow=True).start()
        else:
            return stream_writer.trigger(processingTime= processing_time).start()
        
    def upsert_workouts(self, once=True, processing_time = '5 seconds', startingVersion = 0):
        from pyspark.sql import functions as f
        from pyspark.sql import types as t

        schema = t.StructType([
                t.StructField('user_id', t.IntegerType()),
                t.StructField('workout_id', t.IntegerType()),
                t.StructField('timestamp', t.DoubleType()),
                t.StructField('action', t.StringType()),
                t.StructField('session_id', t.IntegerType())
                ])
        

        query = f'''merge into {self.catalog}.{self.db_name}.workouts a
                    using workouts_delta b
                    on a.user_id = b.user_id and a.time = b.time
                    when not matched 
                    then insert *'''

        data_upserter = Upserter(query, 'workouts_delta')

        df_delta = (spark.readStream 
                            .option('startingVersion', startingVersion)
                            .option('ignoreDeletes', True)
                            .option('maxFilesPerTrigger', self.maxFilesPerTrigger)
                            #option("badRecordsPath", f"{self.badrecords_path}/")
                            .option('withEventTimeOrder', True)
                            .table(f'{self.catalog}.{self.db_name}.kafka_multiplex_bz')
                            .where(f.col('topic') == 'workout')
                            .select(f.from_json(f.col('value').cast(t.StringType()), schema).alias('val'))
                            .select('val.*')
                            .select('user_id', 'workout_id', 'action', 'session_id', f.col('timestamp').cast(t.TimestampType()).alias('time'))
                            .withWatermark('time', '30 seconds')
                            .dropDuplicates(['user_id', 'time'])
                            )
        write_stream = (df_delta.writeStream
                                    .foreachBatch(data_upserter.upsert)
                                    .outputMode('update')
                                    .option('checkpointLocation', f'{self.checkpoint_base}/workouts')
                                    .queryName('workouts_upsert_stream')
                                    )
        spark.sparkContext.setLocalProperty('spark.scheduler.pool', 'silver_p3')

        if once:
            return write_stream.trigger(availableNow=True).start()
        else:
            return write_stream.trigger(processingTime= processing_time).start()
        
    def age_bins(self, dob_col):
        from pyspark.sql import functions as f
        age_col = f.floor(f.months_between(f.current_date(), dob_col)/12).alias('age')
        return (f.when(age_col < 18, 'under_18')
                    .when((age_col >= 18) & (age_col < 25), '18-25')
                    .when((age_col >= 25) & (age_col < 35), '25-35')
                    .when((age_col >= 35) & (age_col < 45), '35-45')
                    .when((age_col >= 45) & (age_col < 55), '45-55')
                    .when((age_col >= 55) & (age_col < 65), '55-65')
                    .when((age_col >= 65) & (age_col < 75), '65-75')
                    .when((age_col >= 75) & (age_col < 85), '75-85')
                    .when((age_col >= 85) & (age_col < 95), '85-95')
                    .when(age_col> 95, '95+')
                    .otherwise('invalid age').alias('age')
                    )
        
    def upsert_user_bins(self, once=True, processing_time = '5 seconds', startingVersion = 0):
        from pyspark.sql import functions as f

        query = f'''merge into {self.catalog}.{self.db_name}.user_bins a
                    using user_bins_delta b
                    on a.user_id = b.user_id
                    when matched
                    then update set *
                    when not matched 
                    then insert *'''

        data_upserter = Upserter(query, 'user_bins_delta')

        df_user = spark.table(f"{self.catalog}.{self.db_name}.users").select("user_id")
        

        df_delta = (spark.readStream
                         .option("startingVersion", startingVersion)
                         .option("ignoreChanges", True)
                         .option("withEventTimeOrder", "true")
                         .option("maxFilesPerTrigger", self.maxFilesPerTrigger)
                         .table(f"{self.catalog}.{self.db_name}.user_profile")
                         .join(df_user, ["user_id"], "left")
                         .select('user_id', self.age_bins(f.col('dob')), 'gender', 'city', 'state')
                         
                   )
        stream_writer = (df_delta.writeStream
                                .foreachBatch(data_upserter.upsert)
                                .outputMode('update')
                                .option('checkpointLocation', f'{self.checkpoint_base}/user_bins')
                                .queryName("user_bins_upsert_stream")
                        )
        
        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "silver_p3")

        if once == True:
            return stream_writer.trigger(availableNow=True).start()
        else:
            return stream_writer.trigger(processingTime=processing_time).start()

    def upsert_completed_workouts(self, once=True, processing_time="15 seconds", startingVersion=0):
        from pyspark.sql import functions as f

        query = f'''merge into {self.catalog}.{self.db_name}.completed_workouts a
                    using completed_workouts_delta b
                    on a.user_id = b.user_id and a.workout_id = b.workout_id and a.session_id = b.session_id
                    when not matched
                    then insert *'''

        data_upserter=Upserter(query, "completed_workouts_delta")

        df_start = (spark.readStream
                         .option("startingVersion", startingVersion)
                         .option("ignoreDeletes", True)
                         .option("withEventTimeOrder", "true")
                         .option("maxFilesPerTrigger", self.maxFilesPerTrigger)
                         .table(f"{self.catalog}.{self.db_name}.workouts")
                         .where(f.col('action') == 'start')
                         .select("user_id", "workout_id", "session_id", f.col('time').alias('start_time'))
                         .withWatermark("start_time", "30 seconds")
                         .dropDuplicates(["user_id", "workout_id", "session_id", "start_time"])
                         )
        df_stop = (spark.readStream
                         .option("startingVersion", startingVersion)
                         .option("ignoreDeletes", True)
                         .option("withEventTimeOrder", "true")
                         .option("maxFilesPerTrigger", self.maxFilesPerTrigger)
                         .table(f"{self.catalog}.{self.db_name}.workouts")
                         .where(f.col('action') == 'stop')
                         .select("user_id", "workout_id", "session_id", f.col('time').alias('end_time'))
                         .withWatermark("end_time", "30 seconds")
                         .dropDuplicates(["user_id", "workout_id", "session_id", "start_time"])
                         )

        join_condition =  [df_start.user_id == df_stop.user_id, df_start.workout_id==df_stop.workout_id, 
                           df_start.session_id==df_stop.session_id, df_stop.end_time < df_start.start_time + f.expr('interval 3 hour')]


        df_delta = (df_start.join(df_stop, join_condition)
                        .select(df_start.user_id, df_start.workout_id, df_start.session_id, df_start.start_time, df_stop.end_time))
        
        stream_writer = (df_delta.writeStream
                                 .foreachBatch(data_upserter.upsert)
                                 .outputMode("append")
                                 .option("checkpointLocation", f"{self.checkpoint_base}/completed_workouts")
                                 .queryName("completed_workouts_upsert_stream")
                        )

        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "silver_p1")
        
        if once == True:
            return stream_writer.trigger(availableNow=True).start()
        else:
            return stream_writer.trigger(processingTime=processing_time).start()
        
    def upsert_workout_bpm(self, once=True, processing_time="15 seconds", startingVersion=0):
        from pyspark.sql import functions as f


        query = f"""
        MERGE INTO {self.catalog}.{self.db_name}.workout_bpm a
        USING workout_bpm_delta b
        ON a.user_id = b.user_id AND a.workout_id = b.workout_id AND a.session_id = b.session_id AND a.time = b.time
        WHEN NOT MATCHED THEN INSERT *
        """
        
        data_upserter=Upserter(query, "workout_bpm_delta")        
        
        df_users = spark.read.table("users")

        df_completed_workouts = (spark.readStream
                                      .option("startingVersion", startingVersion)
                                      .option("ignoreDeletes", True)
                                      .option("withEventTimeOrder", "true")
                                      .option("maxFilesPerTrigger", self.maxFilesPerTrigger)
                                      .table(f"{self.catalog}.{self.db_name}.completed_workouts")
                                      .join(df_users, "user_id")
                                      .selectExpr("user_id", "device_id", "workout_id", "session_id", "start_time", "end_time")
                                      .withWatermark("end_time", "30 seconds")
                                 )
        
        df_bpm = (spark.readStream
                       .option("startingVersion", startingVersion)
                       .option("ignoreDeletes", True)
                       .option("withEventTimeOrder", "true")
                       .option("maxFilesPerTrigger", self.maxFilesPerTrigger)
                       .table(f"{self.catalog}.{self.db_name}.heart_rate")
                       .filter("valid = True")                         
                       .select("device_id", "time", "heartrate")
                       .withWatermark("time", "30 seconds")
                   )
      
        join_condition = [df_completed_workouts.device_id == df_bpm.device_id, 
                          df_bpm.time > df_completed_workouts.start_time, df_bpm.time <= df_completed_workouts.end_time,
                          df_completed_workouts.end_time < df_bpm.time + f.expr('interval 3 hour')] 
        
        df_delta = (df_bpm.join(df_completed_workouts, join_condition)
                          .select("user_id", "workout_id","session_id", "start_time", "end_time", "time", "heartrate")
                   )
        
        stream_writer = (df_delta.writeStream
                                 .foreachBatch(data_upserter.upsert)
                                 .outputMode("append")
                                 .option("checkpointLocation", f"{self.checkpoint_base}/workout_bpm")
                                 .queryName("workout_bpm_upsert_stream")
                        )

        spark.sparkContext.setLocalProperty("spark.scheduler.pool", "silver_p2")
        
        if once == True:
            return stream_writer.trigger(availableNow=True).start()
        else:
            return stream_writer.trigger(processingTime=processing_time).start()
        
    def await_queries(self, once):
        if once:
            for stream in spark.streams.active:
                stream.awaitTermination()

    def upsert(self, once=True, processing_time="5 seconds"):
        import time
        start = int(time.time())
        print(f"Executing silver layer upsert ...")
        self.upsert_users(once, processing_time)
        self.upsert_gym_logs(once, processing_time)
        self.upsert_user_profile(once, processing_time)
        self.upsert_workouts(once, processing_time)
        self.upsert_heart_rate(once, processing_time)        
        self.await_queries(once)
        print(f"Completed silver layer 1 upsert {round((time.time() - start)/60,2)} mins")
        self.upsert_user_bins(once, processing_time)
        self.upsert_completed_workouts(once, processing_time)        
        self.await_queries(once)
        print(f"Completed silver layer 2 upsert {round((time.time() - start)/60,2)} mins")
        self.upsert_workout_bpm(once, processing_time)
        self.await_queries(once)
        print(f"Completed silver layer 3 upsert {round((time.time() - start)/60,2)} mins")

    def assert_count(self, table_name, expected_count, filter="true"):
        print(f"Validating record counts in {table_name}...", end='')
        actual_count = spark.read.table(f"{self.catalog}.{self.db_name}.{table_name}").where(filter).count()
        assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where {filter}" 
        print(f"Found {actual_count:,} / Expected {expected_count:,} records where {filter}: Success")

    def validate(self, sets):
        import time
        start = int(time.time())
        print(f"\nValidating silver layer records...")
        self.assert_count("users", 5 if sets == 1 else 10)
        self.assert_count("gym_logs", 8 if sets == 1 else 16)
        self.assert_count("user_profile", 5 if sets == 1 else 10)
        self.assert_count("workouts", 16 if sets == 1 else 32)
        self.assert_count("heart_rate", sets * 253801)
        self.assert_count("user_bins", 5 if sets == 1 else 10)
        self.assert_count("completed_workouts", 8 if sets == 1 else 16)
        self.assert_count("workout_bpm", 3900 if sets == 1 else 8100)
        print(f"Silver layer validation completed in {int(time.time()) - start} seconds")